In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
DATA_PROCESSED = ROOT / "data_processed"

in_path = DATA_PROCESSED / "01_score_state_features.parquet"
assert in_path.exists(), f"File not found: {in_path}"

df = pd.read_parquet(in_path)

print("Loaded df shape:", df.shape)

Loaded df shape: (48211, 33)


Additional features for determining server and point winner.

In [3]:
df["is_p1_server"] = (df["PointServer"] == 1).astype("int8")
df["server_won_point"] = (df["PointWinner"] == df["PointServer"]).astype("int8")

Per match serve probabilities.

In [5]:
import numpy as np

df["p1_serving"] = (df["PointServer"] == 1).astype("int8")
df["p2_serving"] = (df["PointServer"] == 2).astype("int8")

df["p1_srv_points_won"] = ((df["PointServer"] == 1) & (df["PointWinner"] == 1)).astype("int8")
df["p2_srv_points_won"] = ((df["PointServer"] == 2) & (df["PointWinner"] == 2)).astype("int8")

per_match = (
    df.groupby(["match_id", "player1", "player2", "slam", "year"], as_index=False)
      .agg(
          p1_srv_points=("p1_serving", "sum"),
          p2_srv_points=("p2_serving", "sum"),
          p1_srv_points_won=("p1_srv_points_won", "sum"),
          p2_srv_points_won=("p2_srv_points_won", "sum"),
      )
)

per_match["p_point_srv1"] = per_match["p1_srv_points_won"] / per_match["p1_srv_points"]
per_match["p_point_srv2"] = per_match["p2_srv_points_won"] / per_match["p2_srv_points"]

print(per_match.shape)
per_match.head()


(335, 11)


,match_id,player1,player2,slam,year,p1_srv_points,p2_srv_points,p1_srv_points_won,p2_srv_points_won,p_point_srv1,p_point_srv2
0,2011-ausopen-2501,Caroline Wozniacki,Francesca Schiavone,ausopen,2011,85,97,49,49,0.576471,0.505155
1,2011-ausopen-2502,Andrea Petkovic,Na Li,ausopen,2011,56,52,26,35,0.464286,0.673077
2,2011-ausopen-2503,Agnieszka Radwanska,Kim Clijsters,ausopen,2011,84,67,43,39,0.511905,0.582090
3,2011-ausopen-2504,Petra Kvitova,Vera Zvonareva,ausopen,2011,55,51,23,31,0.418182,0.607843
4,2011-ausopen-2601,Caroline Wozniacki,Na Li,ausopen,2011,98,101,52,56,0.530612,0.554455


In [7]:
OUT_DIR = ROOT / "data_processed"
out_path = OUT_DIR / "02_serve_state_features.parquet"
df.to_parquet(out_path, index=False)

print("Saved:", out_path)
print("Shape:", df.shape)

Saved: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_processed/02_serve_state_features.parquet
Shape: (48211, 39)
